# 02 — Fetch isochrones

Asks Mapbox, for each open branch, "what area can you reach in N minutes?" and
stores the answer as a polygon.

| | |
|---|---|
| **Reads** | `clean/branches.csv`, `features/isochrones.parquet` (its own cache) |
| **Writes** | `features/isochrones.parquet` |
| **Needs** | `cpl-s3`, `mapbox-token` |

**One notebook, three nodes.** `MODE` and `TRAVEL_TIME` come from the
environment, so the same file can sit on the canvas three times with different
values — that is the point of this step as an Elyra exercise.

**No Feast here.** The KFP version checks a Feast online store before calling
Mapbox. Elyra nodes cannot mount the ConfigMap that Feast needs for its client
config, so this version does the same freshness check against the parquet cache
in S3 instead. Same behaviour, one less moving part.

### Running this as an Elyra node

Drag this notebook onto a pipeline canvas, then open its **node properties**:

| Property | Value |
|---|---|
| Runtime image | Datascience with Python 3.12 (or any image with pip access) |
| Kubernetes secret | `cpl-s3` → `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_S3_ENDPOINT`, `AWS_S3_BUCKET` |
| Kubernetes secret | `mapbox-token` → `MAPBOX_TOKEN` |
| Environment variables | `MODE=driving`, `TRAVEL_TIME=5` (per node) |

The notebook installs its own dependencies in the first cell, so any runtime
image works. Everything it reads and writes lives in S3 — nothing is passed
between nodes as a file, so node ordering is the only dependency.

In [ ]:
%pip install -q boto3 requests shapely pandas pyarrow

In [ ]:
import io
import os
import logging

import boto3
import pandas as pd

logging.basicConfig(level=logging.INFO, force=True)
log = logging.getLogger("fetch_isochrones")

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["AWS_S3_ENDPOINT"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    verify=False,
)
BUCKET = os.environ["AWS_S3_BUCKET"]
log.info("Bucket: %s", BUCKET)

## Parameters

Change these inline when you are running interactively. When this runs as an
Elyra node the environment wins, so each node on the canvas can target a
different mode.

In [ ]:
MODE = os.environ.get("MODE", "driving")
TRAVEL_TIME = int(os.environ.get("TRAVEL_TIME", "5"))

# Skip any branch fetched more recently than this. Traffic does not move much
# inside an hour, and the Mapbox free tier is 100k requests/month.
FRESHNESS_HOURS = float(os.environ.get("FRESHNESS_HOURS", "1"))

RATE_LIMIT_DELAY = 0.25  # Mapbox allows 300 req/min
MAX_RETRIES = 3

MAPBOX_TOKEN = os.environ["MAPBOX_TOKEN"]
CACHE_KEY = "features/isochrones.parquet"

log.info("Mode: %s, travel time: %d min", MODE, TRAVEL_TIME)

## Load branches

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key="clean/branches.csv")
branches = pd.read_csv(io.BytesIO(obj["Body"].read()))
branches = branches[branches["PERMANENTLY_CLOSED"] == 0].reset_index(drop=True)
log.info("Loaded %d active branches", len(branches))

branches[["BRANCH", "LATITUDE", "LONGITUDE"]].head()

## What is already cached?

The cache is append-only — every fetch is kept so you can see how a branch's
reachable area changes over time. For the freshness check only the newest row
per `(branch, mode, travel_time)` matters.

In [ ]:
from datetime import datetime, timezone, timedelta

now = datetime.now(timezone.utc)
cutoff_ts = int((now - timedelta(hours=FRESHNESS_HOURS)).timestamp())

try:
    obj = s3.get_object(Bucket=BUCKET, Key=CACHE_KEY)
    existing = pd.read_parquet(io.BytesIO(obj["Body"].read()))
except Exception as e:
    log.info("No cache yet (%s)", type(e).__name__)
    existing = pd.DataFrame()

fresh_branches = set()
if not existing.empty:
    mine = existing[(existing["mode"] == MODE) & (existing["travel_time"] == TRAVEL_TIME)]
    if not mine.empty:
        newest = mine.groupby("branch")["fetch_ts"].max()
        fresh_branches = set(newest[newest >= cutoff_ts].index)

log.info("Cache holds %d rows; %d branches are still fresh for %s/%dmin",
         len(existing), len(fresh_branches), MODE, TRAVEL_TIME)

## Fetch

One request per stale branch. Three tries each, because a single timeout
should not cost the whole run.

In [ ]:
import time

import requests
from shapely.geometry import shape

base_url = f"https://api.mapbox.com/isochrone/v1/mapbox/{MODE}/"

new_rows = []
stats = {"cached": 0, "fetched": 0, "failed": 0}

for _, branch in branches.iterrows():
    branch_name = branch["BRANCH"]
    lat, lon = branch["LATITUDE"], branch["LONGITUDE"]

    if branch_name in fresh_branches:
        stats["cached"] += 1
        continue

    success = False
    for attempt in range(MAX_RETRIES):
        try:
            url = (
                f"{base_url}{lon},{lat}"
                f"?contours_minutes={TRAVEL_TIME}"
                f"&polygons=true&access_token={MAPBOX_TOKEN}"
            )
            resp = requests.get(url, timeout=30)
            resp.raise_for_status()
            geometry = resp.json()["features"][0]["geometry"]

            new_rows.append({
                "isochrone_key": f"{branch_name}|{MODE}|{TRAVEL_TIME}",
                "branch": branch_name,
                "mode": MODE,
                "travel_time": TRAVEL_TIME,
                "latitude": lat,
                "longitude": lon,
                "geometry_wkt": shape(geometry).wkt,
                "fetch_ts": int(now.timestamp()),
                "event_timestamp": now,
            })
            stats["fetched"] += 1
            success = True
            break
        except Exception as e:
            log.warning("Attempt %d/%d failed for %s (%s, %dmin): %s",
                        attempt + 1, MAX_RETRIES, branch_name, MODE, TRAVEL_TIME, e)
            if attempt < MAX_RETRIES - 1:
                time.sleep(1)

    if not success:
        stats["failed"] += 1
        log.error("All %d attempts failed for %s (%s, %dmin)",
                  MAX_RETRIES, branch_name, MODE, TRAVEL_TIME)

    time.sleep(RATE_LIMIT_DELAY)

log.info("Cached: %(cached)d, Fetched: %(fetched)d, Failed: %(failed)d", stats)

## Append to the cache

In [ ]:
if new_rows:
    new_df = pd.DataFrame(new_rows)
    new_df["event_timestamp"] = pd.to_datetime(new_df["event_timestamp"], utc=True)
    result = pd.concat([existing, new_df], ignore_index=True) if not existing.empty else new_df
else:
    result = existing

if not result.empty:
    buf = io.BytesIO()
    result.to_parquet(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=BUCKET, Key=CACHE_KEY, Body=buf.getvalue())

log.info("Done (%s, %dmin). Cache now holds %d rows.", MODE, TRAVEL_TIME, len(result))
result.groupby(["mode", "travel_time"]).size()